# 예제 02. 검색 품질 측정 — Recall@k, MRR

전산실험 · AI Agent 만들기 · **10주차** · 예상 소요 35분

## 목표
- **검색을 답변과 분리해서** 측정한다
- Recall@1, Recall@3, MRR 을 계산한다
- 청킹 전략을 바꿔 가며 비교한다
- 어휘 검색과 의미 검색을 비교한다

> **검색이 답변 품질의 상한을 정합니다.**
> Recall@3 가 60% 면, 나머지 40% 는 아무리 좋은 모델을 써도 근거 없이 답합니다.

### 준비 — 공용 모듈과 데이터 만들기 / Setup — write the shared modules and data

이 수업 저장소에는 노트북(`.ipynb`)만 둡니다. 그래서 실습에 필요한 공용 모듈과
데이터를 **아래 셀들이 직접 파일로 만들어 냅니다.** 내려받지 않으므로 네트워크가
느려도, 저장소가 비공개로 바뀌어도 그대로 동작합니다.

아래 셀을 **위에서부터 차례로 한 번씩** 실행한 뒤 다음으로 넘어가세요.
내용을 지금 읽을 필요는 없습니다. 만들어진 `.py` 파일은 왼쪽 파일 탐색기에서
언제든 열어 볼 수 있습니다.

The course repository contains notebooks only. The cells below **write the shared
modules and data files themselves** instead of downloading them. Run them once,
top to bottom, then continue. You do not need to read them now — the generated
`.py` files can be opened from the file browser on the left.

In [ ]:
%%writefile kmu_llm.py
"""전산실험2 - AI Agent 만들기 : 공용 LLM 래퍼.

계명대 통계학과 서버와 Google Gemini를 같은 방식으로 호출한다.
둘 다 OpenAI 호환 API를 제공하므로 openai SDK 하나로 통일하고
base_url 과 model 만 바꾼다.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 이 저장소는 공개(public)이므로 **학과 서버 주소와 API 키를 코드에 적지
 않는다.** 둘 다 수업시간에 공개하고, 각자 Colab Secrets 에 저장해서 쓴다.

     KMU_BASE_URL  = 수업시간에 공개  (예: https://.../v1)
     KMU_API_KEY   = 수업시간에 공개

 왼쪽 열쇠 아이콘(🔑) → 새 secret → 노트북 액세스 토글 켜기
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

사용 예:
    from kmu_llm import LLM, ask, check_server

    check_server()                       # 서버가 살아 있는지 먼저 확인
    print(ask("표준편차를 한 문장으로 설명해줘"))

    llm = LLM(provider="gemini")         # 백업 경로
    print(llm.chat([{"role": "user", "content": "안녕"}]))
"""

from __future__ import annotations

import os
import time
from typing import Any, Iterator

__all__ = [
    "KMU_MODEL",
    "GEMINI_BASE_URL",
    "GEMINI_MODEL",
    "PROVIDERS",
    "LLM",
    "ask",
    "get_secret",
    "get_key",
    "get_base_url",
    "check_server",
]

# --------------------------------------------------------------------------
# 서버 설정
# --------------------------------------------------------------------------
# 학과 서버 주소는 여기에 적지 않는다. Colab Secrets 의 KMU_BASE_URL 에서 읽는다.
# 모델 태그는 공개해도 무방하므로 상수로 둔다.
KMU_MODEL = os.environ.get("KMU_MODEL", "gemma4")

# 백업 경로. Google AI Studio 에서 무료 키를 발급받아 쓴다. (공개 주소)
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEMINI_MODEL = "gemini-2.0-flash"

PROVIDERS: dict[str, dict[str, Any]] = {
    "kmu": {
        "base_url": None,  # 실행 시점에 Secrets 에서 읽는다
        "base_url_key": "KMU_BASE_URL",
        "model": KMU_MODEL,
        "key_name": "KMU_API_KEY",
        "label": "계명대 학과 서버",
    },
    "gemini": {
        "base_url": GEMINI_BASE_URL,
        "base_url_key": None,
        "model": GEMINI_MODEL,
        "key_name": "GEMINI_API_KEY",
        "label": "Google Gemini (백업)",
    },
}


# --------------------------------------------------------------------------
# 비밀정보 가져오기
# --------------------------------------------------------------------------
def get_secret(name: str, *, prompt: bool = True, secret: bool = True) -> str:
    """설정값을 안전한 순서로 찾는다.

    1) Colab Secrets (왼쪽 열쇠 아이콘)  2) 환경변수  3) 직접 입력

    값을 코드에 적지 않는 것이 핵심이다. 노트북이나 모듈을 GitHub 에 올리는
    순간 코드에 적힌 값은 전 세계에 공개된다. API 키뿐 아니라 **내부 서버
    주소도** 마찬가지다.

    Parameters
    ----------
    secret : True 면 입력을 화면에 표시하지 않는다(getpass).
             주소처럼 오타 확인이 중요한 값은 False 로 준다.
    """
    # 1) Colab Secrets
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass

    # 2) 환경변수
    value = os.environ.get(name)
    if value:
        return value.strip()

    # 3) 직접 입력
    if not prompt:
        raise RuntimeError(
            f"'{name}' 를 찾을 수 없습니다.\n"
            f"  Colab 왼쪽 열쇠 아이콘(🔑) → 새 secret → 이름 '{name}'\n"
            f"  값은 수업시간에 공개합니다. 노트북 액세스 토글도 켜세요."
        )

    print(f"[{name}] 를 찾지 못했습니다. 직접 입력해 주세요.")
    print(f"   (Colab Secrets 에 '{name}' 로 저장해 두면 다시 묻지 않습니다)")

    if secret:
        import getpass

        value = getpass.getpass(f"{name}: ").strip()
    else:
        value = input(f"{name}: ").strip()

    if not value:
        raise RuntimeError(f"'{name}' 에 빈 값이 입력되었습니다.")
    os.environ[name] = value  # 같은 런타임 안에서는 다시 묻지 않는다
    return value


def get_key(key_name: str = "KMU_API_KEY", *, prompt: bool = True) -> str:
    """API 키를 가져온다. (get_secret 의 별칭 — 1주차 이름 유지)"""
    return get_secret(key_name, prompt=prompt, secret=True)


def get_base_url(provider: str = "kmu", *, prompt: bool = True) -> str:
    """서버 주소를 가져온다. 끝의 '/' 는 떼고 돌려준다.

    학과 서버 주소는 저장소에 적지 않으므로 Secrets 에서 읽는다.
    '/chat/completions' 까지 붙여서 저장한 경우도 알아서 잘라낸다.
    """
    conf = PROVIDERS[provider]

    if conf["base_url"]:
        return str(conf["base_url"]).rstrip("/")

    url = get_secret(conf["base_url_key"], prompt=prompt, secret=False).rstrip("/")

    # openai SDK 는 base_url 뒤에 /chat/completions 를 스스로 붙인다.
    # 주소를 통째로 저장한 경우를 대비해 정리해 준다.
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]

    conf["base_url"] = url  # 같은 세션에서는 재사용
    return url


# --------------------------------------------------------------------------
# LLM 클라이언트
# --------------------------------------------------------------------------
class LLM:
    """OpenAI 호환 서버를 감싼 얇은 클라이언트.

    Parameters
    ----------
    provider : "kmu" | "gemini"
        어느 서버를 쓸지. 기본은 학과 서버.
    model, base_url, api_key :
        직접 지정하면 기본값을 덮어쓴다. (보통 지정하지 않는다)
    """

    def __init__(
        self,
        provider: str = "kmu",
        model: str | None = None,
        base_url: str | None = None,
        api_key: str | None = None,
        timeout: float = 120.0,
    ) -> None:
        if provider not in PROVIDERS:
            raise ValueError(
                f"알 수 없는 provider: {provider!r}. 가능한 값: {list(PROVIDERS)}"
            )
        conf = PROVIDERS[provider]

        self.provider = provider
        self.label = conf["label"]
        self.model = model or conf["model"]
        self.base_url = base_url or get_base_url(provider)
        self.api_key = api_key or get_key(conf["key_name"])

        try:
            from openai import OpenAI
        except ImportError as exc:  # pragma: no cover
            raise ImportError(
                "openai 패키지가 필요합니다.  !pip install -q openai"
            ) from exc

        self.client = OpenAI(
            api_key=self.api_key, base_url=self.base_url, timeout=timeout
        )

    def __repr__(self) -> str:
        # 주소는 출력하지 않는다. 화면 공유·스크린샷으로 새어나가기 쉽다.
        return f"LLM(provider={self.provider!r}, model={self.model!r})"

    # -- 기본 호출 --------------------------------------------------------
    def raw(self, messages: list[dict[str, str]], **kw: Any):
        """응답 객체 전체를 그대로 돌려준다. usage(토큰 수) 확인용."""
        params: dict[str, Any] = {
            "model": self.model,
            "messages": messages,
            "temperature": 0.7,
            "max_tokens": 1024,
        }
        params.update(kw)
        return self.client.chat.completions.create(**params)

    def chat(self, messages: list[dict[str, str]], **kw: Any) -> str:
        """대화 메시지를 보내고 응답 '텍스트'만 돌려준다."""
        response = self.raw(messages, **kw)
        content = response.choices[0].message.content
        return (content or "").strip()

    def stream(self, messages: list[dict[str, str]], **kw: Any) -> Iterator[str]:
        """응답을 조각(chunk) 단위로 흘려보낸다. 타이핑되는 효과."""
        kw["stream"] = True
        for chunk in self.raw(messages, **kw):
            if not chunk.choices:
                continue
            piece = chunk.choices[0].delta.content
            if piece:
                yield piece

    # -- 편의 메서드 ------------------------------------------------------
    def ask(self, prompt: str, system: str | None = None, **kw: Any) -> str:
        """한 번짜리 질문. messages 를 매번 만들기 번거로울 때 쓴다."""
        messages: list[dict[str, str]] = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        return self.chat(messages, **kw)

    def ask_retry(
        self,
        prompt: str,
        system: str | None = None,
        *,
        n_retry: int = 3,
        wait: float = 2.0,
        **kw: Any,
    ) -> str:
        """일시적인 네트워크·서버 오류를 재시도한다.

        30회 반복 실험처럼 호출이 많은 실습에서 한 번의 실패로 전체가
        멈추지 않도록 하기 위한 것이다.
        """
        last_error: Exception | None = None
        for attempt in range(1, n_retry + 1):
            try:
                return self.ask(prompt, system, **kw)
            except Exception as exc:  # 네트워크·서버 오류 전반
                last_error = exc
                if attempt < n_retry:
                    print(f"   [재시도 {attempt}/{n_retry - 1}] {type(exc).__name__}")
                    time.sleep(wait * attempt)  # 점점 더 오래 기다린다
        raise RuntimeError(f"{n_retry}회 시도 모두 실패했습니다.") from last_error

    def count_tokens(self, messages: list[dict[str, str]]) -> int:
        """입력 토큰 수를 서버에 물어본다. 출력은 1토큰으로 잘라 낭비를 줄인다."""
        return self.raw(messages, max_tokens=1).usage.prompt_tokens


# --------------------------------------------------------------------------
# 모듈 수준 편의 함수
# --------------------------------------------------------------------------
_default_llm: LLM | None = None


def ask(
    prompt: str, system: str | None = None, *, provider: str = "kmu", **kw: Any
) -> str:
    """가장 간단한 한 줄 호출.  ask("안녕") -> "안녕하세요..."

    내부적으로 LLM 객체를 한 번만 만들어 재사용한다.
    """
    global _default_llm
    if _default_llm is None or _default_llm.provider != provider:
        _default_llm = LLM(provider=provider)
    return _default_llm.ask(prompt, system, **kw)


# --------------------------------------------------------------------------
# 서버 진단
# --------------------------------------------------------------------------
def check_server(provider: str = "kmu", base_url: str | None = None) -> dict[str, Any]:
    """서버가 살아 있는지, 어떤 모델이 올라가 있는지 확인한다.

    수업 첫 실습. 실패하면 무엇을 확인해야 하는지 함께 출력한다.

    Returns
    -------
    dict : {"ok": bool, "models": list[str], "error": str | None}
    """
    import requests

    conf = PROVIDERS[provider]
    base_url = (base_url or get_base_url(provider)).rstrip("/")
    api_key = get_key(conf["key_name"])

    # 주소 전체를 출력하지 않는다. 호스트만 가려서 보여준다.
    print(f"조회 중: GET {_mask_url(base_url)}/models")

    try:
        response = requests.get(
            f"{base_url}/models",
            headers={"Authorization": f"Bearer {api_key}"},
            timeout=15,
        )
    except Exception as exc:
        print(f"\n[실패] 서버에 연결하지 못했습니다: {type(exc).__name__}")
        _print_troubleshooting()
        return {"ok": False, "models": [], "error": str(exc)}

    if response.status_code != 200:
        print(f"\n[실패] HTTP {response.status_code}")
        print(f"   응답: {response.text[:300]}")
        if response.status_code in (401, 403):
            print("   -> API 키가 잘못되었거나 만료되었을 가능성이 높습니다.")
        elif response.status_code == 404:
            print("   -> KMU_BASE_URL 이 '/v1' 로 끝나는지 확인하세요.")
        return {"ok": False, "models": [], "error": f"HTTP {response.status_code}"}

    models = [m.get("id", "?") for m in response.json().get("data", [])]

    print(f"\n[성공] 서버가 응답했습니다. 사용 가능한 모델 {len(models)}개:")
    for name in models:
        mark = "  <- 기본 모델" if name == conf["model"] else ""
        print(f"   - {name}{mark}")

    if conf["model"] not in models and models:
        print(
            f"\n[주의] 설정된 모델 '{conf['model']}' 이 목록에 없습니다.\n"
            f"       위 목록에서 골라 LLM(model='...') 로 지정하거나\n"
            f"       환경변수 KMU_MODEL 을 설정하세요."
        )

    return {"ok": True, "models": models, "error": None}


def _mask_url(url: str) -> str:
    """화면 공유·스크린샷 유출을 막기 위해 호스트를 가린다."""
    import re

    return re.sub(r"//([^/]+)", lambda m: "//" + m.group(1)[:4] + "***", url, count=1)


def _print_troubleshooting() -> None:
    """연결 실패 시 확인할 것들."""
    print("\n확인해 볼 것:")
    print("   1. Colab Secrets 의 KMU_BASE_URL 값이 정확합니까?")
    print("      - http / https 를 맞게 적었습니까?")
    print("      - 포트가 필요할 수 있습니다.")
    print("      - 끝이 '/v1' 이어야 합니다. ('/chat/completions' 는 빼도 됩니다)")
    print("   2. 학과 서버가 켜져 있습니까? 교내망에서만 열려 있지 않습니까?")
    print("   3. 다른 주소를 시험해 보려면:")
    print("      check_server(base_url='...')")
    print("   4. 계속 안 되면 백업 경로를 쓰세요:  LLM(provider='gemini')")


In [ ]:
%%writefile rag.py
"""전산실험2 10주차 : RAG — 검색 증강 생성.

9주차에 검정 선택 규칙을 **코드에 직접 적었다.** 잘 동작하지만 한계가 있다.
규칙에 없는 상황이 오면 답할 수 없고, "왜 이 검정인가"를 물으면
"규칙에 그렇게 되어 있어서"라고밖에 말하지 못한다.

이 모듈은 **문서에서 근거를 찾아와 인용**하게 만든다.

    질문 → 검색 → 관련 문서 조각 → 프롬프트에 넣어 답변 → 인용 검증

두 가지 검색 방식을 모두 구현한다.

    TfidfRetriever      단어 겹침으로 찾는다. 설치 불필요, 항상 동작
    EmbeddingRetriever  의미로 찾는다. 서버의 /v1/embeddings 가 필요

둘을 비교하는 것 자체가 실습이다. **어휘 검색과 의미 검색은 잘하는 것이 다르다.**

사용 예:
    from rag import chunk_markdown, TfidfRetriever, answer_with_citations

    chunks = chunk_markdown(open("stat_methods.md", encoding="utf-8").read())
    r = TfidfRetriever(chunks)
    print(answer_with_citations("세 집단 비교에는 뭘 쓰나요?", r, llm))
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Any

__all__ = [
    "Chunk",
    "chunk_markdown",
    "Retriever",
    "TfidfRetriever",
    "EmbeddingRetriever",
    "build_context",
    "check_citations",
    "answer_with_citations",
    "RAG_SYSTEM",
]


# --------------------------------------------------------------------------
# 청킹
# --------------------------------------------------------------------------
@dataclass
class Chunk:
    """검색 단위. 문서를 잘라 놓은 조각 하나."""

    id: int
    title: str
    text: str
    source: str = ""

    @property
    def n_chars(self) -> int:
        return len(self.text)

    def cite(self) -> str:
        return f"[{self.id}] {self.title}"

    def block(self, max_chars: int = 1200) -> str:
        """프롬프트에 넣을 형태."""
        본문 = self.text[:max_chars]
        if len(self.text) > max_chars:
            본문 += " ...(생략)"
        return f"[{self.id}] {self.title}\n{본문}"


def chunk_markdown(
    text: str,
    *,
    heading: str = "## ",
    source: str = "",
    max_chars: int = 2000,
) -> list[Chunk]:
    """마크다운을 제목 단위로 자른다.

    **왜 제목 단위인가**
    고정 길이(예: 500자)로 자르면 문장이 중간에서 끊기고, 한 주제가
    두 조각으로 갈라진다. 문서에 이미 구조가 있으면 그것을 쓰는 편이 낫다.
    "일원분산분석" 절 전체가 한 조각이면 검색이 걸렸을 때 답이 그 안에 다 있다.

    너무 긴 절은 문단 경계에서 다시 나눈다.
    """
    조각: list[Chunk] = []
    현재제목 = "(머리말)"
    버퍼: list[str] = []

    def flush() -> None:
        본문 = "\n".join(버퍼).strip()
        if not 본문:
            return
        if len(본문) <= max_chars:
            조각.append(Chunk(len(조각) + 1, 현재제목, 본문, source))
            return
        # 긴 절은 문단 경계에서 분할
        문단들 = 본문.split("\n\n")
        part, n = [], 0
        for 문단 in 문단들:
            if n + len(문단) > max_chars and part:
                조각.append(Chunk(len(조각) + 1, f"{현재제목} (계속)" if 조각 and
                                조각[-1].title.startswith(현재제목) else 현재제목,
                                "\n\n".join(part), source))
                part, n = [], 0
            part.append(문단)
            n += len(문단)
        if part:
            조각.append(Chunk(len(조각) + 1,
                            f"{현재제목} (계속)" if 조각 and
                            조각[-1].title.startswith(현재제목) else 현재제목,
                            "\n\n".join(part), source))

    for line in text.splitlines():
        if line.startswith(heading):
            flush()
            버퍼 = []
            현재제목 = line[len(heading):].strip()
        else:
            버퍼.append(line)
    flush()

    return [c for c in 조각 if c.n_chars > 40]      # 너무 짧은 조각은 버린다


# --------------------------------------------------------------------------
# 검색기
# --------------------------------------------------------------------------
class Retriever:
    """검색기 공통 인터페이스."""

    name = "base"

    def __init__(self, chunks: list[Chunk]) -> None:
        self.chunks = chunks

    def search(self, query: str, k: int = 3) -> list[tuple[Chunk, float]]:
        raise NotImplementedError

    def show(self, query: str, k: int = 3) -> None:
        print(f"[{self.name}] '{query}'")
        for c, s in self.search(query, k):
            print(f"  {s:.3f}  {c.cite()}")


class TfidfRetriever(Retriever):
    """단어·글자 겹침으로 찾는다 (어휘 검색).

    한국어는 조사가 붙어 단어 경계가 흐릿하므로 **글자 n-gram** 을 쓴다.
    `char_wb` 는 단어 안에서만 n-gram 을 만들어 띄어쓰기를 존중한다.

    장점: 설치·API 불필요, 빠름, 결과를 설명할 수 있음
    단점: **같은 뜻 다른 말**을 못 찾는다 ("세 집단" vs "3개 그룹")
    """

    name = "tfidf"

    def __init__(self, chunks: list[Chunk], *, ngram_range: tuple = (2, 4)) -> None:
        super().__init__(chunks)
        from sklearn.feature_extraction.text import TfidfVectorizer

        self.vectorizer = TfidfVectorizer(
            analyzer="char_wb", ngram_range=ngram_range, min_df=1, sublinear_tf=True
        )
        문서들 = [f"{c.title}\n{c.title}\n{c.text}" for c in chunks]  # 제목에 가중치
        self.matrix = self.vectorizer.fit_transform(문서들)

    def search(self, query: str, k: int = 3) -> list[tuple[Chunk, float]]:
        from sklearn.metrics.pairwise import cosine_similarity

        q = self.vectorizer.transform([query])
        점수 = cosine_similarity(q, self.matrix)[0]
        순서 = 점수.argsort()[::-1][:k]
        return [(self.chunks[i], float(점수[i])) for i in 순서]


class EmbeddingRetriever(Retriever):
    """의미로 찾는다 (의미 검색).

    문장을 벡터로 바꿔 코사인 유사도로 비교한다.
    "세 집단 비교" 와 "3개 그룹의 평균 차이" 를 같은 뜻으로 인식할 수 있다.

    서버에 `/v1/embeddings` 가 없으면 만들 때 예외가 난다.
    `EmbeddingRetriever.available(llm)` 로 먼저 확인할 것.
    """

    name = "embedding"

    def __init__(self, chunks: list[Chunk], llm: Any, *,
                 model: str = "text-embedding-3-small", batch: int = 32) -> None:
        super().__init__(chunks)
        self.llm = llm
        self.model = model
        문서들 = [f"{c.title}\n{c.text}" for c in chunks]
        self.vectors = self._embed(문서들, batch)

    # ----------------------------------------------------------------
    @staticmethod
    def available(llm: Any, model: str = "text-embedding-3-small") -> bool:
        """이 서버가 임베딩을 지원하는가."""
        try:
            llm.client.embeddings.create(model=model, input=["테스트"])
            return True
        except Exception:
            return False

    def _embed(self, texts: list[str], batch: int = 32):
        import numpy as np

        벡터들 = []
        for i in range(0, len(texts), batch):
            덩어리 = texts[i : i + batch]
            응답 = self.llm.client.embeddings.create(model=self.model, input=덩어리)
            벡터들.extend([d.embedding for d in 응답.data])
        arr = np.asarray(벡터들, dtype=float)
        # 정규화해 두면 코사인 유사도가 내적이 된다
        norms = np.linalg.norm(arr, axis=1, keepdims=True)
        return arr / np.clip(norms, 1e-12, None)

    def search(self, query: str, k: int = 3) -> list[tuple[Chunk, float]]:
        q = self._embed([query])[0]
        점수 = self.vectors @ q
        순서 = 점수.argsort()[::-1][:k]
        return [(self.chunks[i], float(점수[i])) for i in 순서]


# --------------------------------------------------------------------------
# 프롬프트 구성과 인용 검증
# --------------------------------------------------------------------------
RAG_SYSTEM = """\
너는 통계 방법론 조언자다. 아래에 주어진 참고 문서에만 근거해서 답한다.

절대 규칙:
- 참고 문서에 없는 내용은 말하지 않는다. 모르면 "참고 문서에 없다"고 답한다.
- 모든 주장 뒤에 근거 번호를 [1] 처럼 붙인다.
- 여러 문서를 쓰면 [1][3] 처럼 모두 표시한다.
- 문서에 없는 번호를 지어내지 않는다.
- 참고 문서와 다른 내용을 아는 것 같아도, 문서를 따른다."""


def build_context(hits: list[tuple[Chunk, float]], max_chars: int = 1200) -> str:
    """검색 결과를 프롬프트에 넣을 문자열로."""
    return "\n\n".join(c.block(max_chars) for c, _ in hits)


_CITATION = re.compile(r"\[(\d+)\]")


@dataclass
class CitationCheck:
    """인용이 제대로 됐는지 검사한 결과."""

    cited: list[int] = field(default_factory=list)
    valid: list[int] = field(default_factory=list)
    invalid: list[int] = field(default_factory=list)   # 존재하지 않는 번호
    unused: list[int] = field(default_factory=list)    # 줬는데 안 쓴 문서
    n_sentences: int = 0
    n_cited_sentences: int = 0

    @property
    def ok(self) -> bool:
        return not self.invalid and bool(self.valid)

    @property
    def coverage(self) -> float:
        """문장 중 몇 %가 근거를 달고 있는가."""
        return self.n_cited_sentences / self.n_sentences if self.n_sentences else 0.0

    def report(self) -> str:
        줄 = [f"인용된 번호: {self.cited}"]
        if self.invalid:
            줄.append(f"⚠ 존재하지 않는 번호: {self.invalid}  ← 지어낸 인용")
        if self.unused:
            줄.append(f"  제공했으나 쓰이지 않은 문서: {self.unused}")
        줄.append(f"  근거를 단 문장 비율: {self.coverage:.0%} "
                  f"({self.n_cited_sentences}/{self.n_sentences})")
        return "\n".join(줄)


def check_citations(answer: str, hits: list[tuple[Chunk, float]]) -> CitationCheck:
    """답변의 인용을 검증한다.

    **RAG에서 가장 중요한 검증이다.**
    모델은 근거 없이 그럴듯한 말을 하거나, 존재하지 않는 문서 번호를
    지어낼 수 있다. 인용이 있다고 해서 믿으면 안 된다.

    여기서 확인하는 것:
      1) 인용한 번호가 실제로 제공된 문서인가
      2) 제공한 문서 중 무엇을 썼는가
      3) 문장 중 몇 %가 근거를 달고 있는가

    확인하지 **못하는** 것: 인용한 내용이 그 문서에 실제로 있는가.
    그것은 사람이 보거나 별도 검증기(12주차 Critic)가 해야 한다.
    """
    제공번호 = {c.id for c, _ in hits}
    인용 = [int(m) for m in _CITATION.findall(answer)]
    고유 = sorted(set(인용))

    문장들 = [s for s in re.split(r"[.!?。]\s*|\n", answer) if len(s.strip()) > 10]
    근거단문장 = sum(1 for s in 문장들 if _CITATION.search(s))

    return CitationCheck(
        cited=고유,
        valid=[i for i in 고유 if i in 제공번호],
        invalid=[i for i in 고유 if i not in 제공번호],
        unused=sorted(제공번호 - set(고유)),
        n_sentences=len(문장들),
        n_cited_sentences=근거단문장,
    )


def answer_with_citations(
    question: str,
    retriever: Retriever,
    llm: Any,
    *,
    k: int = 3,
    max_chars: int = 1200,
    return_details: bool = False,
    **kw: Any,
) -> Any:
    """검색 → 답변 → 인용 검증까지 한 번에.

    Returns
    -------
    str, 또는 return_details=True 면
    {"answer": ..., "hits": ..., "check": CitationCheck}
    """
    kw.setdefault("temperature", 0.0)
    kw.setdefault("max_tokens", 700)

    hits = retriever.search(question, k=k)
    context = build_context(hits, max_chars)

    prompt = (
        f"## 참고 문서\n{context}\n\n"
        f"## 질문\n{question}\n\n"
        f"위 참고 문서에만 근거해서 답하라. 모든 주장에 [번호] 를 붙여라."
    )
    answer = llm.chat(
        [{"role": "system", "content": RAG_SYSTEM},
         {"role": "user", "content": prompt}], **kw
    )
    check = check_citations(answer, hits)

    if return_details:
        return {"answer": answer, "hits": hits, "check": check, "context": context}
    return answer


In [ ]:
!pip install -q openai

# 이 수업 저장소에서 공용 모듈 내려받기

from kmu_llm import LLM, ask, check_server

PROVIDER = "kmu"          # 학과 서버가 안 되면 "gemini" 로 바꾸세요
llm = LLM(provider=PROVIDER)
print("준비 완료:", llm)

In [ ]:
%%writefile stat_methods.md
# 통계 방법론 참고 문서

전산실험2 10주차 RAG 실습용 지식베이스.
에이전트가 검정을 고를 때 **근거로 인용할 문서**입니다.

각 절이 하나의 청크가 됩니다. 절 제목을 검색 단서로 삼도록 작성했습니다.

---

## 독립표본 t검정 (independent samples t-test)

서로 다른 두 집단의 평균을 비교할 때 쓴다. 예를 들어 치료군과 대조군의
혈압 평균이 다른지 확인하는 경우다.

**전제 조건**
1. 두 집단이 서로 독립이다. 같은 대상을 두 번 측정한 것이 아니다.
2. 결과변수가 연속형이다.
3. 각 집단에서 결과변수가 정규분포를 따른다.
4. 두 집단의 분산이 같다 (등분산). 이 가정이 깨지면 Welch 보정을 쓴다.

**등분산 가정에 대하여**
등분산 검정(Levene, Bartlett)을 먼저 하고 t검정을 고르는 2단계 절차는
1종 오류율을 부풀린다는 비판이 있다. 현대적 권고는 **처음부터 Welch t검정을
쓰는 것**이다. Welch는 등분산일 때도 성능 손실이 거의 없다.
scipy에서는 `ttest_ind(a, b, equal_var=False)`이다.

**정규성이 깨지면**
표본이 각 집단 30 이상이면 중심극한정리로 t검정이 비교적 견고하다.
표본이 작거나 치우침이 심하면 Mann-Whitney U 검정을 쓴다.

**효과크기**
Cohen's d를 보고한다. 0.2 작음, 0.5 중간, 0.8 큼이 관례적 기준이지만
분야마다 다르므로 맥락에서 해석해야 한다. p값만 보고하지 말 것.

---

## 대응표본 t검정 (paired samples t-test)

같은 대상을 두 번 측정한 값을 비교할 때 쓴다. 치료 전후 혈압,
같은 학생의 사전·사후 점수 같은 경우다.

**독립표본과 혼동하지 말 것**
가장 흔한 실수다. 같은 사람에게서 나온 두 값은 독립이 아니다.
독립표본 t검정을 쓰면 개인차가 오차에 섞여 들어가 검정력을 잃는다.
설계가 대응인지 독립인지는 **데이터가 아니라 실험 설계**가 결정한다.

**전제 조건**
1. 짝지어진 관측이다.
2. 두 값의 **차이**가 정규분포를 따른다. 각 값이 정규일 필요는 없다.
3. 차이가 연속형이다.

**정규성이 깨지면**
Wilcoxon 부호순위 검정을 쓴다. 이상치 하나가 차이의 정규성을 깨뜨리는
경우가 흔하므로, 정규성 검정 전에 이상치를 먼저 확인해야 한다.

**효과크기**
Cohen's dz = 차이의 평균 / 차이의 표준편차.
또는 차이의 중앙값과 신뢰구간을 보고한다.

---

## Mann-Whitney U 검정

독립표본 t검정의 비모수 대안이다. 정규성 가정이 깨졌거나 결과변수가
순서형일 때 쓴다.

**무엇을 검정하는가**
흔한 오해가 있다. Mann-Whitney는 "중앙값이 같다"를 검정하는 것이 **아니다**.
두 분포에서 무작위로 하나씩 뽑았을 때 한쪽이 클 확률이 1/2인지를 본다.
두 분포의 모양이 같다고 가정할 수 있을 때만 중앙값 비교로 해석할 수 있다.

**전제 조건**
1. 두 집단이 독립이다.
2. 결과변수가 최소 순서형이다.
3. (중앙값 해석을 원하면) 두 분포의 모양이 비슷하다.

**효과크기**
순위이연상관(rank-biserial correlation)을 보고한다.

---

## Wilcoxon 부호순위 검정 (signed-rank test)

대응표본 t검정의 비모수 대안이다. 차이의 정규성이 깨졌을 때 쓴다.

**전제 조건**
1. 짝지어진 관측이다.
2. 차이의 분포가 0을 중심으로 대칭이다.

두 번째 조건이 깨지면 부호검정(sign test)을 고려한다.
부호검정은 대칭성을 요구하지 않지만 검정력이 낮다.

**차이가 0인 짝**
차이가 정확히 0인 관측은 순위를 매길 수 없다. scipy는 기본적으로
제외한다(`zero_method='wilcox'`). 그런 짝이 많으면 결과 해석에 주의한다.

---

## 일원분산분석 (one-way ANOVA)

세 집단 이상의 평균을 비교할 때 쓴다.

**두 집단에 쓰지 말 것**
두 집단이면 t검정을 쓴다. 수학적으로 F = t²이라 결과는 같지만,
효과의 방향을 알 수 없고 신뢰구간을 얻기 어렵다.

**왜 t검정을 여러 번 하면 안 되는가**
세 집단에 t검정을 세 번 하면 적어도 하나에서 1종 오류가 날 확률이
0.05가 아니라 약 0.14로 오른다. 이것을 다중비교 문제라고 한다.
ANOVA는 "적어도 하나의 집단이 다르다"를 한 번에 검정한다.

**전제 조건**
1. 집단이 서로 독립이다.
2. 각 집단에서 결과변수가 정규분포를 따른다.
3. 집단 간 분산이 같다 (등분산). Levene 검정으로 확인한다.

**등분산이 깨지면**
Welch ANOVA를 쓴다. `scipy.stats`에는 없고 `pingouin.welch_anova`나
직접 구현이 필요하다. 또는 Kruskal-Wallis를 쓴다.

**유의하면 그 다음**
ANOVA는 "어딘가 다르다"만 알려준다. 어느 집단끼리 다른지는
사후검정(post-hoc)으로 확인한다. Tukey HSD가 표준적이다.
사후검정 없이 "B가 A보다 높다"고 말하면 안 된다.

**효과크기**
에타제곱(eta squared) 또는 오메가제곱을 보고한다.

---

## Kruskal-Wallis 검정

일원분산분석의 비모수 대안이다. 세 집단 이상, 정규성이 깨졌을 때 쓴다.

**전제 조건**
1. 집단이 독립이다.
2. 결과변수가 최소 순서형이다.

**유의하면 그 다음**
Dunn 검정으로 사후 비교를 한다. 다중비교 보정(Bonferroni, Holm)을
반드시 적용한다.

**효과크기**
엡실론제곱(epsilon squared)을 보고한다.

---

## 카이제곱 독립성 검정 (chi-square test of independence)

두 범주형 변수가 서로 연관되어 있는지 검정한다.
흡연 여부와 질병 발생의 관계 같은 경우다.

**전제 조건**
1. 관측이 서로 독립이다. 같은 사람이 두 번 세어지면 안 된다.
2. **모든 칸의 기대빈도가 5 이상**이다. 이것이 가장 자주 깨지는 조건이다.

**기대빈도가 5 미만이면**
2×2 표에서는 Fisher 정확검정을 쓴다. 더 큰 표에서는 범주를 합치거나
몬테카를로 방법으로 p값을 구한다. `scipy.stats.chi2_contingency`가
반환하는 `expected` 배열을 반드시 확인할 것.

**연속성 보정**
2×2 표에서 Yates 연속성 보정을 적용할지에 대해 논쟁이 있다.
scipy는 기본적으로 적용한다(`correction=True`). 보수적이라는 비판이 있으므로
보고할 때 어느 쪽을 썼는지 명시한다.

**대응 설계라면**
같은 대상을 두 번 측정한 범주형 자료(치료 전후 개선 여부 등)에는
McNemar 검정을 쓴다. 카이제곱을 쓰면 독립 가정이 깨진다.

**효과크기**
Cramer's V를 보고한다. 표 크기에 따라 해석 기준이 다르다.

---

## 피어슨 상관 (Pearson correlation)

두 연속형 변수의 **선형** 관계 강도를 잰다.

**전제 조건**
1. 두 변수가 모두 연속형이다.
2. 관계가 선형이다. 산점도를 반드시 먼저 그려 볼 것.
3. 이변량 정규성. 특히 이상치에 매우 민감하다.

**흔한 오해**
상관계수 0은 "관계가 없다"가 아니라 "**선형** 관계가 없다"는 뜻이다.
U자 관계는 상관이 0에 가깝게 나온다.

**상관은 인과가 아니다**
교란변수가 있을 수 있고, 인과의 방향이 반대일 수 있다.
보고할 때 인과적 표현("영향을 준다", "때문이다")을 쓰지 않는다.

**표본 크기와 유의성**
표본이 크면 r = 0.05 같은 미미한 상관도 통계적으로 유의해진다.
p값과 함께 **r 값 자체와 신뢰구간**을 보고해야 한다.

---

## 스피어만 상관 (Spearman correlation)

순위에 기반한 상관이다. 단조 관계를 잰다.

**언제 쓰는가**
1. 정규성이 깨졌을 때
2. 한쪽 또는 양쪽이 순서형일 때
3. 이상치의 영향을 줄이고 싶을 때
4. 관계가 단조이지만 선형이 아닐 때

**피어슨과의 차이**
피어슨은 선형 관계, 스피어만은 단조 관계를 잰다.
지수적으로 증가하는 관계는 스피어만이 1에 가깝지만 피어슨은 그보다 낮다.

---

## 정규성 검정 (Shapiro-Wilk 등)

**표본 크기의 함정**
정규성 검정은 표본 크기에 민감하다.
- 표본이 작으면(n < 30) 검정력이 낮아 웬만한 위배를 잡지 못한다.
- 표본이 크면(n > 500) 실질적으로 무시할 만한 위배도 유의하게 나온다.

따라서 **p값만 보고 기계적으로 판단하면 안 된다.**
Q-Q plot과 히스토그램을 함께 보고, 왜도·첨도를 확인한다.

**무엇의 정규성인가**
t검정에서 필요한 것은 **원 데이터가 아니라 표본평균의 분포**가 정규인 것이다.
중심극한정리에 의해 표본이 커지면 원 분포가 정규가 아니어도 성립한다.
대응표본에서는 **차이**의 정규성을 본다.

**Shapiro-Wilk vs Kolmogorov-Smirnov**
Shapiro-Wilk가 일반적으로 검정력이 높다. KS 검정은 모수를 추정해서
쓰면 보수적이 되므로 Lilliefors 보정이 필요하다.

---

## 다중비교 보정

여러 검정을 동시에 수행하면 적어도 하나에서 1종 오류가 날 확률이 오른다.
k번 검정하면 그 확률은 대략 1 − (1 − α)^k 이다.

**Bonferroni**
α를 검정 횟수로 나눈다. 가장 단순하고 가장 보수적이다.
검정이 많으면 검정력을 크게 잃는다.

**Holm**
순차적 Bonferroni. Bonferroni보다 검정력이 높으면서 같은 수준으로
집단별 오류율을 통제한다. 기본 선택으로 권장한다.

**Benjamini-Hochberg (FDR)**
거짓발견률을 통제한다. 탐색적 분석에서 검정이 수백 개일 때 적합하다.
유전체 연구 등에서 표준이다.

**언제 보정하는가**
사전에 정한 주 가설 하나에는 보정하지 않는다.
사후검정, 여러 결과변수, 여러 하위집단 분석에는 보정한다.

---

## 효과크기와 신뢰구간

**p값만으로는 부족하다**
p값은 "효과가 없다면 이 정도 결과가 나올 확률"이지 효과의 크기가 아니다.
표본이 크면 미미한 차이도 유의해진다.

**항상 함께 보고할 것**
1. 효과크기 (Cohen's d, eta squared, Cramer's V 등)
2. 신뢰구간 (보통 95%)
3. 표본 크기

**"유의하지 않다"의 의미**
유의하지 않다는 것은 "차이가 없다"는 증거가 **아니다**.
"이 표본으로는 차이를 확인하지 못했다"는 뜻이다.
차이가 없음을 보이려면 동등성 검정(equivalence test)이 필요하다.

보고할 때 "차이가 없다"고 쓰지 말고
"차이가 있다고 하기 어렵다" 또는 "유의한 차이가 관찰되지 않았다"고 쓴다.

---

## 결측치 처리

**완전제거법 (listwise deletion)**
결측이 하나라도 있는 행을 통째로 뺀다. 가장 단순하다.
결측이 완전 무작위(MCAR)일 때만 편향이 없다.
결측이 많으면 표본을 크게 잃는다.

**단일 대체 (mean/median imputation)**
평균이나 중앙값으로 채운다. 분산을 과소추정하게 되어
표준오차가 작아지고 1종 오류가 늘어난다. **권장하지 않는다.**

**다중대체 (multiple imputation)**
결측을 여러 번 다르게 채워 여러 데이터셋을 만들고,
각각 분석한 뒤 결과를 합친다. 불확실성을 제대로 반영한다.
`statsmodels.imputation.mice` 또는 R의 `mice` 패키지.

**보고할 것**
결측이 몇 건이었는지, 어떻게 처리했는지, 결측 메커니즘에 대한 가정을
반드시 명시한다.

---

## 이상치 처리

**탐지 기준**
- IQR 기준: Q1 − 1.5×IQR 미만 또는 Q3 + 1.5×IQR 초과.
  탐색용으로는 유용하지만 정규분포에서도 약 0.7%가 걸린다.
- 3×IQR을 쓰면 극단값만 잡힌다.
- z점수 3 이상: 정규성을 가정하며 이상치 자체가 평균과 표준편차를
  오염시키므로 순환 문제가 있다.

**IQR 기준의 가정**
IQR 기준은 대칭 분포를 암묵적으로 가정한다. 소득처럼 오른쪽으로
치우친 변수에서는 이상치가 과다 검출된다. 로그 변환 후 판단하거나
분포에 맞는 기준을 쓴다.

**제거할 것인가**
입력 오류가 명백하면(혈압 480) 제거하거나 결측 처리한다.
진짜 관측이라면 함부로 제거하면 안 된다. 제거하면 결과가 어떻게 달라지는지
민감도 분석으로 함께 보고하는 것이 좋다.

**제거 여부는 분석자가 판단할 일이지 자동화 도구가 결정할 일이 아니다.**

---

## 검정 선택 요약표

| 질문 | 결과변수 | 설계 | 정규성 | 검정 |
|---|---|---|---|---|
| 두 집단 평균 비교 | 연속형 | 독립 | 만족 | 독립표본 t (Welch) |
| 두 집단 평균 비교 | 연속형/순서형 | 독립 | 불만족 | Mann-Whitney U |
| 전후 비교 | 연속형 | 대응 | 만족 | 대응표본 t |
| 전후 비교 | 연속형/순서형 | 대응 | 불만족 | Wilcoxon 부호순위 |
| 세 집단 이상 비교 | 연속형 | 독립 | 만족 | 일원분산분석 |
| 세 집단 이상 비교 | 연속형/순서형 | 독립 | 불만족 | Kruskal-Wallis |
| 범주 연관성 | 범주형 | 독립 | — | 카이제곱 (기대빈도 5 이상) |
| 범주 연관성 (작은 표) | 범주형 | 독립 | — | Fisher 정확검정 |
| 범주 연관성 | 범주형 | 대응 | — | McNemar |
| 두 변수 관계 | 연속형 | — | 만족 | 피어슨 상관 |
| 두 변수 관계 | 연속형/순서형 | — | 불만족 | 스피어만 상관 |


In [ ]:
문서 = open("stat_methods.md", encoding="utf-8").read()
print(f"{len(문서):,}자")
print(문서[:400])

In [ ]:
from rag import chunk_markdown, TfidfRetriever

chunks = chunk_markdown(문서)
r = TfidfRetriever(chunks)
print(f"청크 {len(chunks)}개")

In [ ]:
# 골든셋 -- 질문과 '정답 문서 제목'
골든셋 = [
    ("세 집단 이상을 비교하려면 어떤 검정을 쓰나요?",        "일원분산분석"),
    ("정규성이 깨졌는데 세 집단을 비교하려면?",              "Kruskal-Wallis"),
    ("카이제곱 검정의 기대빈도 조건은 무엇인가요?",           "카이제곱"),
    ("2x2 표에서 기대빈도가 5 미만이면 어떻게 하나요?",       "카이제곱"),
    ("치료 전후를 비교하는데 정규성이 깨졌습니다",            "Wilcoxon"),
    ("같은 사람을 두 번 측정한 자료는 어떤 검정인가요?",       "대응표본"),
    ("등분산 가정이 깨지면 t검정을 어떻게 하나요?",           "독립표본"),
    ("상관계수가 0이면 관계가 없다는 뜻인가요?",             "피어슨"),
    ("순서형 변수의 관계를 보려면 어떤 상관을 쓰나요?",        "스피어만"),
    ("유의하지 않다는 것이 차이가 없다는 뜻인가요?",          "효과크기"),
    ("결측치를 평균으로 채워도 되나요?",                    "결측치"),
    ("검정을 여러 번 하면 왜 문제가 되나요?",               "다중비교"),
    ("이상치를 자동으로 제거해도 되나요?",                   "이상치"),
    ("표본이 크면 정규성 검정이 왜 문제인가요?",             "정규성"),
    ("Mann-Whitney 가 중앙값을 검정하는 건가요?",           "Mann-Whitney"),
]
print(f"골든셋 {len(골든셋)}문항")

---
## 1. 세 가지 지표

| 지표 | 뜻 | 언제 보나 |
|---|---|---|
| **Recall@1** | 1등이 정답인 비율 | 문서를 하나만 넣을 때 |
| **Recall@3** | 상위 3개 안에 정답이 있는 비율 | **k=3 으로 넣을 거면 이것** |
| **MRR** | 정답의 역순위 평균 (1등=1, 2등=0.5, 3등=0.33) | 순위 품질 전반 |

In [ ]:
import pandas as pd

def 평가(retriever, 골든셋, k=5):
    행 = []
    for 질문, 정답키워드 in 골든셋:
        hits = retriever.search(질문, k=k)
        순위 = None
        for i, (c, s) in enumerate(hits, 1):
            if 정답키워드 in c.title:
                순위 = i
                break
        행.append({
            "질문": 질문[:30], "정답": 정답키워드,
            "순위": 순위,
            "1등": hits[0][0].title[:26],
            "최고점수": round(hits[0][1], 3),
        })
    df = pd.DataFrame(행)
    # 못 찾으면 None -> pandas 가 NaN 으로 바꾸므로 pd.notna 로 확인한다
    순위 = df["순위"]
    def recall(n):
        return float(순위.apply(lambda x: bool(pd.notna(x) and x <= n)).mean())
    지표 = {
        "Recall@1": recall(1),
        "Recall@3": recall(3),
        "Recall@5": recall(5),
        "MRR": float(순위.apply(lambda x: 1/x if pd.notna(x) else 0.0).mean()),
    }
    return df, 지표

결과df, 지표 = 평가(r, 골든셋)
print("TF-IDF (제목 단위 청킹)")
for k, v in 지표.items():
    print(f"  {k:10s} {v:.3f}")
print()
결과df

In [ ]:
# 틀린 것만 보기
틀린것 = 결과df[결과df["순위"] != 1]
print(f"1등이 아닌 질문 {len(틀린것)}개")
if len(틀린것):
    print(틀린것[["질문", "정답", "순위", "1등"]].to_string(index=False))

> **틀린 것을 보는 것이 지표 숫자보다 중요합니다.**
> 왜 틀렸는지 보면 청킹이나 검색 설정을 어떻게 고칠지 알 수 있습니다.

---
## 2. 청킹 전략 비교

In [ ]:
from rag import Chunk

def 고정길이청크(text, size):
    out = []
    for i in range(0, len(text), size):
        조각 = text[i:i+size]
        # 제목이 있으면 첫 줄을 제목으로 (없으면 앞 20자)
        첫줄 = 조각.strip().split("\n")[0][:40]
        out.append(Chunk(len(out)+1, 첫줄, 조각))
    return out

전략들 = {
    "제목 단위":     chunks,
    "고정 300자":    고정길이청크(문서, 300),
    "고정 800자":    고정길이청크(문서, 800),
    "고정 1500자":   고정길이청크(문서, 1500),
}

비교 = []
for 이름, cs in 전략들.items():
    ret = TfidfRetriever(cs)
    _, m = 평가(ret, 골든셋)
    비교.append({"전략": 이름, "청크수": len(cs),
                "평균길이": int(sum(c.n_chars for c in cs)/len(cs)), **m})

pd.DataFrame(비교).round(3)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

비교df = pd.DataFrame(비교)
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(비교df))
w = 0.25
for i, m in enumerate(["Recall@1", "Recall@3", "MRR"]):
    ax.bar(x + (i-1)*w, 비교df[m], w, label=m)
ax.set_xticks(x); ax.set_xticklabels(비교df["전략"])
ax.set_ylim(0, 1.05); ax.legend(); ax.set_ylabel("score")
ax.set_title("Chunking strategy comparison")
plt.tight_layout(); plt.show()

### 무엇이 보이나

**고정 길이 청킹의 문제**
- 너무 짧으면(300자) 답이 조각 안에 다 없다
- 너무 길면(1500자) 한 조각에 여러 주제가 섞여 검색이 뭉개진다
- **제목이 조각 시작에 안 오면** 제목 가중치를 줄 수도 없다

**문서에 구조가 있으면 그것을 쓰십시오.**

---
## 3. 의미 검색 — 서버가 지원하는가

In [ ]:
from rag import EmbeddingRetriever

지원 = EmbeddingRetriever.available(llm)
print("학과 서버 임베딩 지원:", 지원)

if not 지원:
    print()
    print("이 서버는 /v1/embeddings 를 제공하지 않습니다.")
    print("Gemini 백업 경로로 시도해 볼 수 있습니다:")
    print("   from kmu_llm import LLM")
    print("   emb_llm = LLM(provider='gemini')")
    print("   EmbeddingRetriever.available(emb_llm, model='text-embedding-004')")

In [ ]:
# 지원하면 비교, 아니면 건너뜀
emb = None
if 지원:
    emb = EmbeddingRetriever(chunks, llm)
else:
    try:
        from kmu_llm import LLM
        emb_llm = LLM(provider="gemini")
        if EmbeddingRetriever.available(emb_llm, model="text-embedding-004"):
            emb = EmbeddingRetriever(chunks, emb_llm, model="text-embedding-004")
            print("Gemini 임베딩으로 진행합니다.")
    except Exception as exc:
        print("임베딩 사용 불가:", type(exc).__name__)

if emb is None:
    print()
    print("임베딩을 쓸 수 없습니다. 아래 셀들은 건너뛰고")
    print("TF-IDF 결과만으로 진행하세요. (실습 진행에 지장 없음)")

In [ ]:
if emb is not None:
    _, 지표t = 평가(r, 골든셋)
    _, 지표e = 평가(emb, 골든셋)
    print(pd.DataFrame([{"검색기": "TF-IDF (어휘)", **지표t},
                        {"검색기": "임베딩 (의미)", **지표e}]).round(3).to_string(index=False))
else:
    print("건너뜀")

In [ ]:
# 어휘 검색이 못 찾고 의미 검색이 찾는 경우
바꿔쓴질문 = [
    ("3개 그룹의 평균 차이를 보고 싶어요", "일원분산분석"),
    ("데이터가 정규분포가 아닐 때 두 그룹 비교", "Mann-Whitney"),
    ("빈 칸이 있는 자료는 어떻게 하나요", "결측치"),
    ("검정을 반복하면 생기는 문제", "다중비교"),
]
for q, 정답 in 바꿔쓴질문:
    t_hit = r.search(q, k=1)[0]
    줄 = f"'{q[:28]}'\n    TF-IDF : {t_hit[0].title[:30]:32s} {'O' if 정답 in t_hit[0].title else 'X'}"
    if emb is not None:
        e_hit = emb.search(q, k=1)[0]
        줄 += f"\n    임베딩 : {e_hit[0].title[:30]:32s} {'O' if 정답 in e_hit[0].title else 'X'}"
    print(줄)
    print()

### 어느 쪽이 나은가

**의미 검색이 항상 이기는 것이 아닙니다.**

| | 잘하는 것 | 못하는 것 |
|---|---|---|
| 어휘(TF-IDF) | 고유명사, 정확한 용어, 숫자 | 같은 뜻 다른 말 |
| 의미(임베딩) | 바꿔 쓴 표현, 개념적 유사 | 정확한 용어 매칭 |

`"Shapiro-Wilk"` 를 찾는데 임베딩이 `"정규성 검정"` 을 가져오면 곤란합니다.

> **실무에서는 둘을 합칩니다.** 하이브리드 검색입니다. (연습문제 2)

---
## 4. 문턱값 정하기

In [ ]:
있는질문 = [q for q, _ in 골든셋]
없는질문 = ["생존분석은 언제 쓰나요?", "딥러닝 학습률은 어떻게 정하나요?",
          "베이지안 사후분포란?", "SQL 조인 방법", "논문 투고 절차",
          "R과 파이썬 중 무엇이 낫나요?", "표본 크기는 어떻게 정하나요?"]

있음점수 = [r.search(q, k=1)[0][1] for q in 있는질문]
없음점수 = [r.search(q, k=1)[0][1] for q in 없는질문]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(있음점수, bins=12, alpha=0.6, label="in KB", color="#4C72B0")
ax.hist(없음점수, bins=12, alpha=0.6, label="not in KB", color="#C44E52")
ax.set_xlabel("top-1 similarity"); ax.set_ylabel("count")
ax.set_title("Threshold selection")
ax.legend(); plt.tight_layout(); plt.show()

print(f"있는 질문 최고점수: 중앙값 {np.median(있음점수):.3f}, 최소 {min(있음점수):.3f}")
print(f"없는 질문 최고점수: 중앙값 {np.median(없음점수):.3f}, 최대 {max(없음점수):.3f}")

In [ ]:
# 문턱값별 두 종류 오류
행 = []
for 문턱 in np.arange(0.02, 0.26, 0.02):
    거부한_있음 = sum(1 for s in 있음점수 if s < 문턱)      # 답할 수 있는데 거부
    통과한_없음 = sum(1 for s in 없음점수 if s >= 문턱)      # 없는데 답함
    행.append({"문턱": round(문턱, 2),
              "잘못 거부": f"{거부한_있음}/{len(있음점수)}",
              "잘못 통과": f"{통과한_없음}/{len(없음점수)}"})
pd.DataFrame(행).to_string(index=False)

> **어느 쪽 오류가 더 나쁜가**로 정합니다.
>
> 통계 분석 도구라면 **없는 것을 지어내는 쪽이 훨씬 나쁩니다.**
> 문턱을 높게 잡고 "모른다"를 자주 말하게 하십시오.
> 3주차부터 계속 나오는 이야기입니다.

---
## 연습문제

**문제 1.** 골든셋을 25문항으로 늘리고 Recall@3 의 **신뢰구간**을 구하시오.
현재 n=15 로는 어느 정도 폭인가?

**문제 2.** 하이브리드 검색을 구현하시오.
`점수 = α × TFIDF + (1-α) × 임베딩` 으로 두고 α 를 조정해 최적점을 찾으시오.

**문제 3.** 검색에 실패한 질문들의 공통점을 찾으시오.
질문을 바꾸면 찾아지는가, 문서를 고쳐야 하는가?

**문제 4.** `stat_methods.md` 에 없는 주제(생존분석 등)를 한 절 추가하고
Recall 이 어떻게 변하는지 보시오.

In [ ]:
# 여기에 직접 작성해 보세요

<details>
<summary><b>해설 보기</b></summary>

**문제 1.**
```python
from statsmodels.stats.proportion import proportion_confint
s = int((결과df["순위"] <= 3).sum()); n = len(결과df)
print(proportion_confint(s, n, method="wilson"))
```
n=15 면 구간 폭이 30%p 가까이 됩니다. **"Recall@3 가 0.87 이다"라고 주장할 수 없습니다.**
"0.87, 95% CI [0.62, 0.96]" 이라고 말해야 합니다. 11주차의 주제입니다.

**문제 2.** 점수 스케일이 다르므로 정규화가 필요합니다.
```python
def hybrid(q, k=3, alpha=0.5):
    t = {c.id: s for c, s in r.search(q, k=len(chunks))}
    e = {c.id: s for c, s in emb.search(q, k=len(chunks))}
    tmax, emax = max(t.values()) or 1, max(e.values()) or 1
    합 = {i: alpha*t[i]/tmax + (1-alpha)*e[i]/emax for i in t}
    상위 = sorted(합, key=합.get, reverse=True)[:k]
    return [(chunks[i-1], 합[i]) for i in 상위]
```
실무에서는 점수를 더하는 대신 **순위를 합치는** RRF(Reciprocal Rank Fusion)를
많이 씁니다. 스케일 문제가 없습니다.

</details>

---
## 정리

- **검색을 답변과 분리해서** 측정한다
- 문서 구조를 쓴 청킹이 고정 길이보다 낫다
- 어휘 검색과 의미 검색은 **잘하는 것이 다르다**
- 문턱값은 **두 종류 오류의 교환**. 지어내는 쪽이 더 나쁘다
- 지표 숫자보다 **틀린 것을 보는 것**이 중요하다

**다음 노트북** → [`03_cited_answers.ipynb`](https://colab.research.google.com/github/zoomer1975-boop/ai_agent/blob/main/week10/03_cited_answers.ipynb)